In [1]:
import sys
import os

# Get the directory one level up
parent_dir = os.path.abspath(os.path.join(os.path.dirname("catnipp.ipynb"), '..'))
# Add it to the Python path
sys.path.append(parent_dir)

import copy
import csv
import os
import ray
import torch
import time
from multiprocessing import Pool
import numpy as np
import time
from attention_net import AttentionNet
from runner import Runner
from test_worker import WorkerTest
from test_parameters import *

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS backend for computations.")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU.")


def run_test():
    time0 = time.time()
    if not os.path.exists(result_path):
        os.makedirs(result_path)
    device = torch.device('cuda') if USE_GPU_GLOBAL else torch.device('cpu')
    local_device = torch.device('cuda') if USE_GPU else torch.device('cpu')
    global_network = AttentionNet(INPUT_DIM, EMBEDDING_DIM).to(device)
    checkpoint = torch.load(f'{model_path}/checkpoint.pth')
    global_network.load_state_dict(checkpoint['model'])

    print(f'Loading model: {FOLDER_NAME}...')
    print(f'Total budget range: {BUDGET_RANGE}')

    # init meta agents
    meta_agents = [RLRunner.remote(i) for i in range(NUM_META_AGENT)]
    weights = global_network.to(local_device).state_dict() if device != local_device else global_network.state_dict()
    curr_test = 1
    metric_name = ['budget', 'success_rate', 'RMSE', 'delta_cov_trace', 'MI', 'F1Score', 'cov_trace', 'planning_time']
    perf_metrics = {}
    for n in metric_name:
        perf_metrics[n] = []
    cov_trace_list = []
    time_list = []
    episode_number_list = []
    budget_history = []
    obj_history = []
    obj2_history = []

    try:
        while True:
            jobList = []
            for i, meta_agent in enumerate(meta_agents):
                jobList.append(meta_agent.job.remote(weights, curr_test, budget_range=BUDGET_RANGE, sample_length=SAMPLE_LENGTH))
                curr_test += 1
            done_id, jobList = ray.wait(jobList, num_returns=NUM_META_AGENT)
            done_jobs = ray.get(done_id)

            for job in done_jobs:
                metrics, info = job
                episode_number_list.append(info['episode_number'])
                cov_trace_list.append(metrics['cov_trace'])
                time_list.append(metrics['planning_time'])
                for n in metric_name:
                    perf_metrics[n].append(metrics[n])
                budget_history += metrics['budget_history']
                obj_history += metrics['obj_history']
                obj2_history += metrics['obj2_history']

            if curr_test > NUM_TEST:
                print('#Test sample:', NUM_SAMPLE_TEST, '|#Total test:', NUM_TEST, '|Budget range:', BUDGET_RANGE, '|Sample size:', SAMPLE_SIZE, '|K size:', K_SIZE)
                print('Avg time per test:', (time.time()-time0)/NUM_TEST)
                perf_data = []
                for n in metric_name:
                    perf_data.append(np.nanmean(perf_metrics[n]))
                for i in range(len(metric_name)):
                    print(metric_name[i], ':\t', perf_data[i])
                
                idx = np.array(episode_number_list).argsort()
                cov_trace_list = np.array(cov_trace_list)[idx]
                time_list = np.array(time_list)[idx]

                if SAVE_TRAJECTORY_HISTORY:
                    idx = np.array(budget_history).argsort()
                    budget_history = np.array(budget_history)[idx]
                    obj_history = np.array(obj_history)[idx]
                    obj2_history = np.array(obj2_history)[idx]

                break

        Budget = int(perf_data[0])+1
        if SAVE_CSV_RESULT:
            if TRAJECTORY_SAMPLING:
                csv_filename = f'result/CSV/Budget_'+str(Budget)+'_ts_'+str(PLAN_STEP)+'_'+str(NUM_SAMPLE_TEST)+'_'+str(SAMPLE_SIZE)+'_'+str(K_SIZE)+'_results.csv'
                csv_filename3 = f'result/CSV3/Budget_'+str(Budget)+'_ts_'+str(PLAN_STEP)+'_'+str(NUM_SAMPLE_TEST)+'_'+str(SAMPLE_SIZE)+'_'+str(K_SIZE)+'_planning_time.csv'
            else:
                csv_filename = f'result/CSV/Budget_'+str(Budget)+'_greedy'+'_'+str(SAMPLE_SIZE)+'_'+str(K_SIZE)+'_results.csv'
                csv_filename3 = f'result/CSV3/Budget_'+str(Budget)+'_greedy'+'_'+str(SAMPLE_SIZE)+'_'+str(K_SIZE)+'_planning_time.csv'
            csv_data = [cov_trace_list]
            csv_data3 = [time_list]
            with open(csv_filename, 'a') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerows(csv_data)
            with open(csv_filename3, 'a') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerows(csv_data3)

        if SAVE_TRAJECTORY_HISTORY:
            if TRAJECTORY_SAMPLING:
                csv_filename2 = f'result/CSV2/Budget_'+str(Budget)+'_ts_'+str(PLAN_STEP)+'_'+str(NUM_SAMPLE_TEST)+'_'+str(SAMPLE_SIZE)+'_'+str(K_SIZE)+'_trajectory_result.csv'
            else:
                csv_filename2 = f'result/CSV2/Budget_'+str(Budget)+'_greedy_'+'_'+str(SAMPLE_SIZE)+'_'+str(K_SIZE)+'_trajectory_result.csv'
            new_file = False if os.path.exists(csv_filename2) else True
            field_names = ['budget','obj','obj2']
            with open(csv_filename2, 'a') as csvfile:
                writer = csv.writer(csvfile)
                if new_file:
                    writer.writerow(field_names)
                csv_data = np.concatenate((budget_history.reshape(-1,1), obj_history.reshape(-1,1), obj2_history.reshape(-1,1)), axis=-1)
                writer.writerows(csv_data)

    except KeyboardInterrupt:
        print("CTRL_C pressed. Killing remote workers")
        for a in meta_agents:
            ray.kill(a)


@ray.remote(num_cpus=8/NUM_META_AGENT, num_gpus=NUM_GPU/NUM_META_AGENT)
class RLRunner(Runner):
    def __init__(self, metaAgentID):
        super().__init__(metaAgentID)

    def singleThreadedJob(self, episodeNumber, budget_range, sample_length):
        save_img = True if episodeNumber % SAVE_IMG_GAP == 0 else False
        np.random.seed(SEED + 100 * episodeNumber)
        #torch.manual_seed(SEED + 100 * episodeNumber)
        worker = WorkerTest(self.metaAgentID, self.localNetwork, episodeNumber, budget_range, sample_length, self.device, save_image=save_img, greedy=False, seed=SEED + 100 * episodeNumber)
        worker.work(episodeNumber, 0)
        perf_metrics = worker.perf_metrics
        return perf_metrics

    def multiThreadedJob(self, episodeNumber, budget_range, sample_length):
        save_img = True if (SAVE_IMG_GAP != 0 and episodeNumber % SAVE_IMG_GAP == 0) else False
        #save_img = False
        np.random.seed(SEED + 100 * episodeNumber)
        #torch.manual_seed(SEED + 100 * episodeNumber)
        worker = WorkerTest(self.metaAgentID, self.localNetwork, episodeNumber, budget_range, sample_length, self.device, save_image=save_img, greedy=False, seed=SEED + 100 * episodeNumber)
        subworkers = [copy.deepcopy(worker) for _ in range(NUM_SAMPLE_TEST)]
        p = Pool(processes=NUM_SAMPLE_TEST)
        results = []
        for testID, subw in enumerate(subworkers):
            results.append(p.apply_async(subw.work, args=(episodeNumber, testID+1)))
        p.close()
        p.join()
        all_results = []
        best_score = np.inf
        perf_metrics = None
        for res in results:
            metric = res.get()
            all_results.append(metric)
            if metric['cov_trace'] < best_score: # TODO
                perf_metrics = metric
                best_score = metric['cov_trace']
        return perf_metrics

    def job(self, global_weights, episodeNumber, budget_range, sample_length=None):
        self.set_weights(global_weights)
        metrics = self.singleThreadedJob(episodeNumber, budget_range, sample_length)

        info = {
            "id": self.metaAgentID,
            "episode_number": episodeNumber,
        }

        return metrics, info


if __name__ == '__main__':
    ray.init()
    for i in range(1):
        run_test()


Using MPS backend for computations.


2024-08-09 13:23:45,243	INFO worker.py:1781 -- Started a local Ray instance.


AssertionError: Torch not compiled with CUDA enabled

(raylet) [2024-08-09 13:23:55,176 E 99567 18807567] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2024-08-09_13-23-44_611947_99551 is over 95% full, available space: 14583521280; capacity: 994662584320. Object creation will fail if spilling is required.
(raylet) [2024-08-09 13:24:05,177 E 99567 18807567] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2024-08-09_13-23-44_611947_99551 is over 95% full, available space: 14599344128; capacity: 994662584320. Object creation will fail if spilling is required.
(raylet) [2024-08-09 13:24:15,187 E 99567 18807567] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2024-08-09_13-23-44_611947_99551 is over 95% full, available space: 14616449024; capacity: 994662584320. Object creation will fail if spilling is required.


In [1]:
import sys
import os

# Assuming your notebook is located in the "subfolder"
notebook_dir = os.getcwd()  # Gets the current working directory of the notebook
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)

import copy
import csv
import os
import ray
import torch
import time
from multiprocessing import Pool
import numpy as np
import time
from attention_net import AttentionNet
from runner import Runner
from test_worker import WorkerTest
from test_parameters import *

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS backend for computations.")
else:
    device = torch.device("cpu")
    print("MPS not available, using CPU.")


# Print the parent directory
print("Parent directory:", parent_dir)

try:
    from runner import Runner
    print("Runner imported successfully!")
except ModuleNotFoundError as e:
    print(f"Failed to import Runner: {e}")


def run_test():
    time0 = time.time()
    if not os.path.exists(result_path):
        os.makedirs(result_path)

    # The device is already determined at the start, so we don't need to reassign it here
    local_device = device  # use the same device for local operations

    global_network = AttentionNet(INPUT_DIM, EMBEDDING_DIM).to(device)
    checkpoint = torch.load(f'{model_path}/checkpoint.pth', map_location=device)  # Load checkpoint on the correct device
    global_network.load_state_dict(checkpoint['model'])

    print(f'Loading model: {FOLDER_NAME}...')
    print(f'Total budget range: {BUDGET_RANGE}')

    # Initialize meta agents
    meta_agents = [RLRunner.remote(i) for i in range(NUM_META_AGENT)]
    weights = global_network.to(local_device).state_dict()
    curr_test = 1
    metric_name = ['budget', 'success_rate', 'RMSE', 'delta_cov_trace', 'MI', 'F1Score', 'cov_trace', 'planning_time']
    perf_metrics = {n: [] for n in metric_name}
    cov_trace_list = []
    time_list = []
    episode_number_list = []
    budget_history = []
    obj_history = []
    obj2_history = []

    try:
        while True:
            jobList = []
            for i, meta_agent in enumerate(meta_agents):
                jobList.append(meta_agent.job.remote(weights, curr_test, budget_range=BUDGET_RANGE, sample_length=SAMPLE_LENGTH))
                curr_test += 1
            done_id, jobList = ray.wait(jobList, num_returns=NUM_META_AGENT)
            done_jobs = ray.get(done_id)

            for job in done_jobs:
                metrics, info = job
                episode_number_list.append(info['episode_number'])
                cov_trace_list.append(metrics['cov_trace'])
                time_list.append(metrics['planning_time'])
                for n in metric_name:
                    perf_metrics[n].append(metrics[n])
                budget_history += metrics['budget_history']
                obj_history += metrics['obj_history']
                obj2_history += metrics['obj2_history']

            if curr_test > NUM_TEST:
                print('#Test sample:', NUM_SAMPLE_TEST, '|#Total test:', NUM_TEST, '|Budget range:', BUDGET_RANGE, '|Sample size:', SAMPLE_SIZE, '|K size:', K_SIZE)
                print('Avg time per test:', (time.time() - time0) / NUM_TEST)
                perf_data = [np.nanmean(perf_metrics[n]) for n in metric_name]
                for i, name in enumerate(metric_name):
                    print(name, ':\t', perf_data[i])

                idx = np.array(episode_number_list).argsort()
                cov_trace_list = np.array(cov_trace_list)[idx]
                time_list = np.array(time_list)[idx]

                if SAVE_TRAJECTORY_HISTORY:
                    idx = np.array(budget_history).argsort()
                    budget_history = np.array(budget_history)[idx]
                    obj_history = np.array(obj_history)[idx]
                    obj2_history = np.array(obj2_history)[idx]

                break

        Budget = int(perf_data[0]) + 1
        if SAVE_CSV_RESULT:
            if TRAJECTORY_SAMPLING:
                csv_filename = f'result/CSV/Budget_{Budget}_ts_{PLAN_STEP}_{NUM_SAMPLE_TEST}_{SAMPLE_SIZE}_{K_SIZE}_results.csv'
                csv_filename3 = f'result/CSV3/Budget_{Budget}_ts_{PLAN_STEP}_{NUM_SAMPLE_TEST}_{SAMPLE_SIZE}_{K_SIZE}_planning_time.csv'
            else:
                csv_filename = f'result/CSV/Budget_{Budget}_greedy_{SAMPLE_SIZE}_{K_SIZE}_results.csv'
                csv_filename3 = f'result/CSV3/Budget_{Budget}_greedy_{SAMPLE_SIZE}_{K_SIZE}_planning_time.csv'
            csv_data = [cov_trace_list]
            csv_data3 = [time_list]
            with open(csv_filename, 'a') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerows(csv_data)
            with open(csv_filename3, 'a') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerows(csv_data3)

        if SAVE_TRAJECTORY_HISTORY:
            if TRAJECTORY_SAMPLING:
                csv_filename2 = f'result/CSV2/Budget_{Budget}_ts_{PLAN_STEP}_{NUM_SAMPLE_TEST}_{SAMPLE_SIZE}_{K_SIZE}_trajectory_result.csv'
            else:
                csv_filename2 = f'result/CSV2/Budget_{Budget}_greedy_{SAMPLE_SIZE}_{K_SIZE}_trajectory_result.csv'
            new_file = not os.path.exists(csv_filename2)
            field_names = ['budget', 'obj', 'obj2']
            with open(csv_filename2, 'a') as csvfile:
                writer = csv.writer(csvfile)
                if new_file:
                    writer.writerow(field_names)
                csv_data = np.concatenate((budget_history.reshape(-1, 1), obj_history.reshape(-1, 1), obj2_history.reshape(-1, 1)), axis=-1)
                writer.writerows(csv_data)

    except KeyboardInterrupt:
        print("CTRL_C pressed. Killing remote workers")
        for a in meta_agents:
            ray.kill(a)


@ray.remote(num_cpus=8/NUM_META_AGENT, num_gpus=0)  # Set num_gpus to 0 as MPS doesn't use CUDA
class RLRunner(Runner):
    def __init__(self, metaAgentID):
        import sys
        import os
        parent_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
        sys.path.append(parent_dir)

        # Re-import Runner using the same method as in the main script
        from runner import Runner

        super().__init__(metaAgentID)
        self.device = device  # Use the device defined earlier

    def singleThreadedJob(self, episodeNumber, budget_range, sample_length):
        save_img = episodeNumber % SAVE_IMG_GAP == 0
        np.random.seed(SEED + 100 * episodeNumber)
        worker = WorkerTest(self.metaAgentID, self.localNetwork, episodeNumber, budget_range, sample_length, self.device, save_image=save_img, greedy=False, seed=SEED + 100 * episodeNumber)
        worker.work(episodeNumber, 0)
        perf_metrics = worker.perf_metrics
        return perf_metrics

    def multiThreadedJob(self, episodeNumber, budget_range, sample_length):
        save_img = True if (SAVE_IMG_GAP != 0 and episodeNumber % SAVE_IMG_GAP == 0) else False
        #save_img = False
        np.random.seed(SEED + 100 * episodeNumber)
        #torch.manual_seed(SEED + 100 * episodeNumber)
        worker = WorkerTest(self.metaAgentID, self.localNetwork, episodeNumber, budget_range, sample_length, self.device, save_image=save_img, greedy=False, seed=SEED + 100 * episodeNumber)
        subworkers = [copy.deepcopy(worker) for _ in range(NUM_SAMPLE_TEST)]
        p = Pool(processes=NUM_SAMPLE_TEST)
        results = []
        for testID, subw in enumerate(subworkers):
            results.append(p.apply_async(subw.work, args=(episodeNumber, testID+1)))
        p.close()
        p.join()
        all_results = []
        best_score = np.inf
        perf_metrics = None
        for res in results:
            metric = res.get()
            all_results.append(metric)
            if metric['cov_trace'] < best_score: # TODO
                perf_metrics = metric
                best_score = metric['cov_trace']
        return perf_metrics

    def job(self, global_weights, episodeNumber, budget_range, sample_length=None):
        self.set_weights(global_weights)
        metrics = self.singleThreadedJob(episodeNumber, budget_range, sample_length)

        info = {
            "id": self.metaAgentID,
            "episode_number": episodeNumber,
        }

        return metrics, info


if __name__ == '__main__':
    ray.init()
    for i in range(1):
        run_test()

Using MPS backend for computations.
Parent directory: /Users/joshuaott/Downloads/CAtNIPP-main
Runner imported successfully!


2024-08-09 13:31:06,010	INFO worker.py:1781 -- Started a local Ray instance.
/var/folders/gq/j5vwgp6n61vgbjxq5yr427w40000gn/T/ipykernel_99890/4100591825.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related

Loading model: ipp-4heads...
Total budget range: (9.99999, 10)


(TemporaryActor pid=99913) Exception raised in creation task: The actor died because of an error raised in its creation task, ray::RLRunner.__init__() (pid=99913, ip=127.0.0.1, actor_id=eb1ad3435223c38cd2b3061b01000000, repr=<__main__.FunctionActorManager._create_fake_actor_class.<locals>.TemporaryActor object at 0x106193490>)
(TemporaryActor pid=99913)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(TemporaryActor pid=99913) RuntimeError: The actor with name RLRunner failed to import on the worker. This may be because needed library dependencies are not installed in the worker environment:
(TemporaryActor pid=99913) 
(TemporaryActor pid=99913) ray::RLRunner.__init__() (pid=99913, ip=127.0.0.1, actor_id=eb1ad3435223c38cd2b3061b01000000, repr=<__main__.FunctionActorManager._create_fake_actor_class.<locals>.TemporaryActor object at 0x106193490>)
(TemporaryActor pid=99913)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
(TemporaryActor pid=99913) ModuleNotFoundError: No module named 'runne

ActorDiedError: The actor died because of an error raised in its creation task, [36mray::RLRunner.__init__()[39m (pid=99913, ip=127.0.0.1, actor_id=eb1ad3435223c38cd2b3061b01000000, repr=<__main__.FunctionActorManager._create_fake_actor_class.<locals>.TemporaryActor object at 0x106193490>)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: The actor with name RLRunner failed to import on the worker. This may be because needed library dependencies are not installed in the worker environment:

[36mray::RLRunner.__init__()[39m (pid=99913, ip=127.0.0.1, actor_id=eb1ad3435223c38cd2b3061b01000000, repr=<__main__.FunctionActorManager._create_fake_actor_class.<locals>.TemporaryActor object at 0x106193490>)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'runner'

(raylet) [2024-08-09 13:31:16,002 E 99899 18819744] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2024-08-09_13-31-05_391003_99890 is over 95% full, available space: 17967247360; capacity: 994662584320. Object creation will fail if spilling is required.
(raylet) [2024-08-09 13:31:26,100 E 99899 18819744] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2024-08-09_13-31-05_391003_99890 is over 95% full, available space: 17971376128; capacity: 994662584320. Object creation will fail if spilling is required.
(raylet) [2024-08-09 13:31:36,198 E 99899 18819744] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2024-08-09_13-31-05_391003_99890 is over 95% full, available space: 17951522816; capacity: 994662584320. Object creation will fail if spilling is required.
(raylet) [2024-08-09 13:31:46,292 E 99899 18819744] (raylet) file_system_monitor.cc:111: /tmp/ray/session_2024-08-09_13-31-05_391003_99890 is over 95% full, available space: 17951436800; capacity: 994662584320. Obj

In [13]:
%pip install shapely

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.7 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.
